# Packages

In [1]:
import pickle

import pandas as pd
from pandas.testing import assert_frame_equal

# Loading Dependencies

## Datasets

In [2]:
df_events = pd.read_csv("../data/events.csv")
df_products = pd.read_csv("../data/products.csv")

## Scaler

In [3]:
with open("../model/model.pkl", "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]
scaler = artifact["scaler"]
feature_cols = artifact["feature_cols"]

# Helper Functions

In [4]:
def compute_user_top_affinity_category(
    events: pd.DataFrame,
    products: pd.DataFrame,
) -> pd.DataFrame:
    """Compute each user's top-affinity product category.

    The top-affinity category is the one with the most event rows after
    joining events with products on product_id. Ties are broken by
    interaction count (descending), then category name (ascending).

    Args:
        events: User interaction events.
        products: Product catalog with category metadata.

    Returns:
        DataFrame with columns ``user_id`` and ``top_affinity_category``.
    """
    events_with_category = events.merge(
        products[["product_id", "category"]],
        on="product_id",
        how="inner",
    )
    category_counts = (
        events_with_category.groupby(["user_id", "category"], as_index=False)
        .size()
        .rename(columns={"size": "interaction_count"})
    )
    top_affinity = (
        category_counts.sort_values(
            ["user_id", "interaction_count", "category"],
            ascending=[True, False, True],
        )
        .drop_duplicates("user_id")
        .rename(columns={"category": "top_affinity_category"})[
            ["user_id", "top_affinity_category"]
        ]
    )
    return top_affinity


In [5]:
def is_cold_start_user(user_id: str, events: pd.DataFrame) -> bool:
    """Check whether a user has no interaction history.

    Args:
        user_id: Target user identifier.
        events: User interaction events.

    Returns:
        ``True`` when the user does not appear in ``events``.
    """
    return user_id not in set(events["user_id"].unique())

In [6]:
def build_features(
    events: pd.DataFrame,
    products: pd.DataFrame,
    feature_columns: list[str],
    pairs: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Build model features for user-product pairs.

    Cold-start users (no rows in ``events``) receive imputed
    ``interactions = 0`` and ``user_affinity_match = 0``. Product-level
    features (``price``, ``avg_rating``, ``popularity_score``) come from
    ``products.csv`` as usual.

    Args:
        events: User interaction events.
        products: Product catalog.
        feature_columns: Ordered feature names expected by the model.
        pairs: Optional user-product pairs to score. Defaults to every
            distinct pair observed in ``events``.

    Returns:
        DataFrame with ``user_id``, ``product_id``, ``is_cold_start``,
        and model features.
    """
    if pairs is None:
        pairs = events[["user_id", "product_id"]].drop_duplicates()

    interactions = (
        events.groupby(["user_id", "product_id"], as_index=False)
        .size()
        .rename(columns={"size": "interactions"})
    )

    product_features = products[
        ["product_id", "category", "price", "avg_rating", "popularity_score"]
    ]

    top_affinity = compute_user_top_affinity_category(events, products)
    users_with_history = set(events["user_id"].unique())

    features = (
        pairs.merge(interactions, on=["user_id", "product_id"], how="left")
        .merge(product_features, on="product_id", how="left")
        .merge(top_affinity, on="user_id", how="left")
    )

    features["is_cold_start"] = ~features["user_id"].isin(users_with_history)
    cold_start_mask = features["is_cold_start"]
    known_user_mask = ~cold_start_mask

    features.loc[known_user_mask, "interactions"] = (
        features.loc[known_user_mask, "interactions"].fillna(0).astype(int)
    )
    features.loc[known_user_mask, "user_affinity_match"] = (
        features.loc[known_user_mask, "category"]
        == features.loc[known_user_mask, "top_affinity_category"]
    ).astype(int)

    features.loc[cold_start_mask, "interactions"] = 0
    features.loc[cold_start_mask, "user_affinity_match"] = 0

    return features[
        ["user_id", "product_id", "is_cold_start", *feature_columns]
    ]

In [7]:
def build_features_for_user(
    user_id: str,
    events: pd.DataFrame,
    products: pd.DataFrame,
    feature_columns: list[str],
    candidate_product_ids: list[str] | None = None,
) -> pd.DataFrame:
    """Build features for one user against candidate products.

    Args:
        user_id: Target user identifier.
        events: User interaction events.
        products: Product catalog.
        feature_columns: Ordered feature names expected by the model.
        candidate_product_ids: Product ids to score. Defaults to the full
            catalog.

    Returns:
        Feature DataFrame for the user-product pairs.
    """
    if candidate_product_ids is None:
        candidate_product_ids = products["product_id"].tolist()

    pairs = pd.DataFrame(
        {
            "user_id": user_id,
            "product_id": candidate_product_ids,
        }
    )

    return build_features(events, products, feature_columns, pairs=pairs)

In [8]:
def scale_features(
    features: pd.DataFrame,
    scaler,
    feature_columns: list[str],
) -> pd.DataFrame:
    """Scale raw features with the fitted training scaler.

    Args:
        features: DataFrame containing raw model features.
        scaler: Fitted ``StandardScaler`` from the model artifact.
        feature_columns: Ordered feature names used during training.

    Returns:
        DataFrame of scaled features with the same index as ``features``.
    """
    X = features[feature_columns].astype(float).fillna(0).to_numpy()
    X_scaled = scaler.transform(X)

    return pd.DataFrame(
        X_scaled,
        columns=feature_columns,
        index=features.index,
    )

In [9]:
def recommend_for_user(
    user_ids: str | list[str],
    events: pd.DataFrame,
    products: pd.DataFrame,
    model,
    scaler,
    feature_columns: list[str],
    top_n: int | None = None,
) -> pd.DataFrame:
    """Generate ranked product recommendations for one or more users.

    Cold-start users bypass the model: features are imputed as
    ``interactions = 0`` and ``user_affinity_match = 0``, and products are
    ranked by ``popularity_score``. Known users are ranked by the model
    purchase propensity score (``predict_proba``).

    Args:
        user_ids: Target user identifier or list of user identifiers.
        events: User interaction events.
        products: Product catalog.
        model: Trained purchase propensity model.
        scaler: Fitted feature scaler from the model artifact.
        feature_columns: Ordered feature names expected by the model.
        top_n: Optional number of recommendations to return per user.

    Returns:
        DataFrame with recommendations for all requested users, sorted by
        ``user_id``, ``recommendation_score``, and ``product_id``.
    """
    if isinstance(user_ids, str):
        user_ids = [user_ids]

    users_with_history = set(events["user_id"].unique())
    cold_start_users = [user_id for user_id in user_ids if user_id not in users_with_history]
    known_users = [user_id for user_id in user_ids if user_id in users_with_history]

    candidate_product_ids = products["product_id"].tolist()
    recommendation_frames: list[pd.DataFrame] = []

    if cold_start_users:
        cold_start_pairs = pd.DataFrame(
            [
                {"user_id": user_id, "product_id": product_id}
                for user_id in cold_start_users
                for product_id in candidate_product_ids
            ]
        )
        cold_start_recommendations = build_features(
            events,
            products,
            feature_columns,
            pairs=cold_start_pairs,
        )
        cold_start_recommendations["recommendation_score"] = (
            cold_start_recommendations["popularity_score"]
        )
        recommendation_frames.append(cold_start_recommendations)

    if known_users:
        known_pairs = pd.DataFrame(
            [
                {"user_id": user_id, "product_id": product_id}
                for user_id in known_users
                for product_id in candidate_product_ids
            ]
        )

        known_recommendations = build_features(
            events,
            products,
            feature_columns,
            pairs=known_pairs,
        )
        scaled_features = scale_features(known_recommendations, scaler, feature_columns)
        known_recommendations["recommendation_score"] = model.predict_proba(
            scaled_features.to_numpy()
        )[:, 1]
        recommendation_frames.append(known_recommendations)

    recommendations = pd.concat(recommendation_frames, ignore_index=True)
    recommendations = recommendations.sort_values(
        ["user_id", "recommendation_score", "product_id"],
        ascending=[True, False, True],
    )

    if top_n is not None:
        recommendations = (
            recommendations.groupby("user_id", group_keys=False)
            .head(top_n)
            .reset_index(drop=True)
        )

    return recommendations

# Derivating Feature Variables

In [10]:
df = build_features(df_events, df_products, feature_cols)

In [11]:
df

,user_id,product_id,is_cold_start,interactions,price,avg_rating,popularity_score,user_affinity_match
0,u_0231,p_042,False,1,278.10,4.0,0.068,0.0
1,u_0078,p_012,False,6,752.81,4.9,0.096,1.0
2,u_0322,p_059,False,2,104.16,4.5,0.386,1.0
3,u_0121,p_045,False,4,273.64,3.1,0.327,1.0
4,u_0316,p_036,False,1,621.59,4.0,0.262,1.0
...,...,...,...,...,...,...,...,...
5281,u_0084,p_030,False,1,622.35,3.8,0.801,1.0
5282,u_0063,p_057,False,1,353.48,4.5,0.326,0.0
5283,u_0162,p_044,False,1,262.57,3.6,0.127,1.0
5284,u_0363,p_025,False,1,443.30,3.9,0.323,0.0


In [12]:
df

,user_id,product_id,is_cold_start,interactions,price,avg_rating,popularity_score,user_affinity_match
0,u_0231,p_042,False,1,278.10,4.0,0.068,0.0
1,u_0078,p_012,False,6,752.81,4.9,0.096,1.0
2,u_0322,p_059,False,2,104.16,4.5,0.386,1.0
3,u_0121,p_045,False,4,273.64,3.1,0.327,1.0
4,u_0316,p_036,False,1,621.59,4.0,0.262,1.0
...,...,...,...,...,...,...,...,...
5281,u_0084,p_030,False,1,622.35,3.8,0.801,1.0
5282,u_0063,p_057,False,1,353.48,4.5,0.326,0.0
5283,u_0162,p_044,False,1,262.57,3.6,0.127,1.0
5284,u_0363,p_025,False,1,443.30,3.9,0.323,0.0


# Scaling Features

In [13]:
df_scaled = scale_features(df, scaler, feature_cols)

df_model_input = pd.concat(
    [df[["user_id", "product_id"]], df_scaled],
    axis=1,
)

df_model_input.head()

,user_id,product_id,interactions,price,avg_rating,popularity_score,user_affinity_match
0,u_0231,p_042,-0.594659,-0.513768,0.013679,-1.250936,-1.351569
1,u_0078,p_012,5.196365,1.539539,1.520086,-1.071331,0.739881
2,u_0322,p_059,0.563546,-1.266127,0.850572,0.788868,0.739881
3,u_0121,p_045,2.879955,-0.533060,-1.492728,0.410414,0.739881
4,u_0316,p_036,-0.594659,0.971961,0.013679,-0.006527,0.739881


# Testing Model

In [45]:
known_user_ids = df_events["user_id"].unique().tolist()

df_local_preds = recommend_for_user(
    known_user_ids,
    df_events,
    df_products,
    model,
    scaler,
    feature_cols,
    top_n=120,
).sort_values(["user_id", "product_id"], ascending=False)
df_local_preds = df_local_preds.reset_index(drop=True)
df_local_preds["interactions"] = df_local_preds["interactions"].astype(int)
df_local_preds["user_affinity_match"] = df_local_preds["user_affinity_match"].astype(int)

In [38]:
# retrieving from predic model cluster
df_ecs_preds = pd.read_csv("s3://personalization-data-272175292064/predictions/predictions_20260729105850.csv")

In [39]:
list(df_local_preds.columns)

['user_id',
 'product_id',
 'is_cold_start',
 'interactions',
 'price',
 'avg_rating',
 'popularity_score',
 'user_affinity_match',
 'recommendation_score']

In [41]:
df_ecs_preds = df_ecs_preds.rename(columns={"purchase_proba": "recommendation_score"}).sort_values(["user_id", "product_id"], ascending=False)
df_ecs_preds["is_cold_start"] = False
df_ecs_preds = df_ecs_preds.reset_index(drop=True)
df_ecs_preds = df_ecs_preds[
    ['user_id',
    'product_id',
    'is_cold_start',
    'interactions',
    'price',
    'avg_rating',
    'popularity_score',
    'user_affinity_match',
    'recommendation_score']
]

In [46]:
assert_frame_equal(df_local_preds, df_ecs_preds)

# Cold Start

In [15]:
most_popular_product = df_products.sort_values(
    ["popularity_score", "product_id"],
    ascending=[False, True],
).iloc[0]

most_popular_product

product_id           p_030
category            beleza
price               622.35
avg_rating             3.8
popularity_score     0.801
Name: 30, dtype: object

In [16]:
batch_user_ids = [
    "u_cold_start",
    df_events["user_id"].iloc[0],
]

recommend_for_user(
    batch_user_ids,
    df_events,
    df_products,
    model,
    scaler,
    feature_cols,
    top_n=1,
)

,user_id,product_id,is_cold_start,interactions,price,avg_rating,popularity_score,user_affinity_match,recommendation_score
0,u_0231,p_032,False,3.0,24.31,3.2,0.125,1.0,0.206472
1,u_cold_start,p_030,True,0.0,622.35,3.8,0.801,0.0,0.801000
